# သင်ခန်းစာ ၀၄ - ကိရိယာ အသုံးပြုမှု ဒီဇိုင်းပုံစံ

ဤသင်ခန်းစာတွင် Microsoft Agent Framework (Python) ကို အသုံးပြု၍ AI အေးဂျင့်များအတွက် **ကိရိယာ အသုံးပြုမှု** ဒီဇိုင်းပုံစံကို သင်ယူမည်ဖြစ်သည်။ ကျွန်ုပ်တို့က ဖော်ပြပါအချက်များ ပါဝင်သည် -

- `@tool` decorator နှင့် type ပုံစံပါတဲ့ parameter များဖြင့် function ကိရိယာများဖော်ပြခြင်း
- မော်ဒယ် အသိပေးမှုပေးရန် ကိရိယာ schema များ ပေးခြင်း
- `approval_mode` ဖြင့် ကိရိယာ အကောင်အထည်ဖော်မှု ထိန်းအုပ်ခြင်း
- Pydantic မော်ဒယ်များနှင့် `response_format` မှတစ်ဆင့် **ဖွဲ့စည်းထားသော output** ပြန်လည်တင်ပြခြင်း

အဆိုပါအခြေအနေမှာ အရောက်အပို့ အကြံပေး အေးဂျင့်တစ်ဦးဖြစ်ပြီး ခရီးသွားရောက်ရာများကို ရှာဖွေ၊ ရရှိနိုင်မှု စစ်ဆေး၊ နှင့် လေကြောင်းအချက်အလက်များကို ရယူနိုင်သည်။


## တပ်ဆင်မှု


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool Decorator ဖြင့် ကိရိယာများ ကို သတ်မှတ်ခြင်း

`@tool` decorator သည် ရိုးရိုး Python function ကို agent တစ်ဦး က အသုံးပြုနိုင်သည့် ကိရိယာအဖြစ် ပြောင်းလဲပေးသည်။
အဓိကအချက်များမှာ -

- **docstring** သည် မော်ဒယ်မြင်ရသော ကိရိယာရွေ့ပြောင်းချက် ဖြစ်လာသည်။
- **Type annotations** (ဖော်ပြချက်များပါသော `Annotated` အပါအဝင်) က ကိရိယာ schema ကို သတ်မှတ်ပေးသည်။
- `approval_mode` သည် အသုံးပြုသူမှ ခေါ်ဆိုမှုတိုင်းကို အတည်ပြုရန် မလိုအပ်ပုံ သို့မဟုတ် လိုအပ်ပုံကို ထိန်းချုပ်ပေးသည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## အကြောင်းအရာစုံသော Agent တစ်ခု ဖန်တီးခြင်း

Model သည် အသုံးပြုသူ၏ မေးခွန်းကို ဖြေဆိုရာတွင် လိုအပ်သည့် tools များကို သုံးနိုင်ရန်အတွက် တိုက်ရိုက် client ထံသို့ tools သုံးချက်ကို အကုန်ပေးပို့ခြင်း။


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## စနစ်တကျ ထုတ်လွှင့်မှုနှင့် ကိရိယာများ

`response_format` ကို Pydantic မော်ဒယ်တစ်ခုအဖြစ် သတ်မှတ်ခြင်းက နေရာလွတ် စာသားမဟုတ်ဘဲ ကောင်းမွန်စွာအမျိုးအစားသတ်မှတ်ထားသော JSON အရာဝတ္ထုကို ရရှိစေရန် ဖြစ်သည်။ ၎င်းသည် အောက်ပိုင်းကုဒ်များမှ ရလဒ်ကို ပရိုဂရမ်ဖြင့် စားသုံးရန်လိုအပ်သောအခါ အသုံးဝင်ပါသည်။


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ကိရိယာ အတည်ပြုမှု ပုံစံများ

`@tool` ပေါ်ရှိ `approval_mode` ပရမေတားသည် ကိရိယာခေါ်ဆိုမှုများ သည် ဆောင်ရွက်မှုမတိုင်မှီ လူသား၏ အတည်ပြုမှု လိုအပ်မလိုကိုထိန်းချုပ်သည်။

| အမျိုးအစား | အပြုအမူ |
|---|---|
| `"never_require"` | ကိရိယာသည် အလိုအလျောက် ပြေးဆွဲမည် — အသုံးပြုသူ အတည်ပြုချက် မလိုအပ်ပါ။ |
| `"always_require"` | အတိုင်းအတာတိုင်းကို ဆောင်ရွက်ရန် မတိုင်မှီ အသုံးပြုသူ၏ အတည်ပြုချက် လိုအပ်သည်။ |

သက်ရောက်မှုများ ရှိသော ကိရိယာများတွင် (ဥပမာ - လေယာဉ် ခုံကြပ်ခြင်း၊ ခရက်ဒစ်ကတ် ချာ့ဂ်ခြင်း) `"always_require"` ကို သုံးပါ၊ ထို့ကြောင့် လူသားတစ်ဦးက လုပ်ငန်းစဉ်အတွင်းတွင် ရှိနေပါမည်။


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

ဤသင်ခန်းစာတွင် သင်သိရှိခဲ့သည်မှာ-

1. **@tool** decorator ကို typed parameters နှင့် tool schema အဖြစ် မှတ်ချက်စာသားများဖြင့် အသုံးပြုကာ ကိရိယာများကို ဖေါ်ပြခြင်း။
2. **အတန်းလိုက်လုပ်ဆောင်နိုင်ရန်** အမျိုးမျိုးသောကိရိယာများကို ပေါင်းစပ်ခြင်း။
3. **ဖန်တီးထားသော output** ကို Pydantic မော်ဒယ်အား `response_format` အဖြစ် ပေးပို့ခြင်း။
4. **မဟုတ်ခင် လူပေါင်းကူညီမှုရှိရန်** `approval_mode` ဖြင့် ကိရိယာအသုံးပြုမှု ထိန်းချုပ်ခြင်း။

ဤနည်းစနစ်များသည် ထိရောက်၍ အပြင်စနစ်များနှင့်လုံခြုံစွာ ဆက်သွယ်နိုင်သော အသင့်တော်သော agent များ တည်ဆောက်ရန် အခြေခံတစ်ခုဖြစ်ပါသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ချက်ကြားချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ တိကျမှုအတွက် ကျွန်ုပ်တို့ကြိုးစားပေမယ့် အလိုအလျောက်ဘာသာပြန်မှုများတွင် အမှားအယွင်းများ ဖြစ်ပေါ်နိုင်ကြောင်း သတိပြုပါရန် မေတ္တာရပ်ခံအပ်ပါသည်။ မူရင်းစာတမ်းကို မူလဘာသာဖြင့်သာ အတည်ပြုရမည့် ထောက်ခံရင်းမြစ်အဖြစ် သတ်မှတ်ရမည်ဖြစ်သည်။ အရေးကြီးသောအချက်အလက်များအတွက် လူ့ဘာသာပြန်ပညာရှင်မှ ဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်မှုကို အသုံးပြုရာမှ ဖြစ်ပေါ်နိုင်သည့် နားလည်မှုမှားခြင်းများအတွက် ကျွန်ုပ်တို့ မည်သည့်တာဝန်ဖြင့်မဆို မခံယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
